# 01 — Data Preparation (Binance Raw Klines)
Download **full 12-column** klines from `data.binance.vision` (includes
`taker_buy_base_vol` and `num_trades` — critical for Market Microstructure).

Then resample to 1H and 4H, clean, and save to parquet on Google Drive.

In [ ]:
# Install dependencies
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tensorboard tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# Clone/update repo from GitHub (local Colab filesystem — fast)
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)
print(f'Using project at {REPO_DIR}')

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')

# Output dirs on Google Drive for persistence
config.data.processed_dir = '/content/drive/MyDrive/scalp2/data/processed'
os.makedirs(config.data.processed_dir, exist_ok=True)

print(f'Symbol: {config.data.symbol}')
print(f'Date range: {config.data.date_range.start} to {config.data.date_range.end}')
print(f'Timeframes: {config.data.timeframes.primary}, {config.data.timeframes.mtf}')

In [ ]:
# ============================================================
#  BINANCE DATA VISION DOWNLOADER (Full 12-Column Klines)
# ============================================================
import requests
import zipfile
import io
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from pathlib import Path

BINANCE_VISION_BASE = 'https://data.binance.vision/data/futures/um/daily/klines'
SYMBOL = 'BTCUSDT'
INTERVAL = '15m'

KLINE_COLUMNS = [
    'open_time_ms', 'open', 'high', 'low', 'close', 'volume',
    'close_time_ms', 'quote_asset_volume', 'number_of_trades',
    'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
]

CSV_PATH = '/content/drive/MyDrive/scalp2/data/btcusdt_15min_full.csv'

def download_binance_vision_day(symbol, interval, date_str):
    """Download a single day's kline ZIP from data.binance.vision."""
    url = f'{BINANCE_VISION_BASE}/{symbol}/{interval}/{symbol}-{interval}-{date_str}.zip'
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code != 200:
            return None
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            csv_name = zf.namelist()[0]
            with zf.open(csv_name) as f:
                # Force all columns to string first, we'll convert later
                df = pd.read_csv(f, header=None, names=KLINE_COLUMNS, dtype=str)
        # Drop any row where open_time_ms is not a pure number (header rows)
        df = df[pd.to_numeric(df['open_time_ms'], errors='coerce').notna()]
        return df
    except Exception:
        return None

# Determine date range
start_date = pd.to_datetime(config.data.date_range.start).date()
end_date = pd.to_datetime(config.data.date_range.end).date()

# Check for existing data
if os.path.exists(CSV_PATH):
    df_existing = pd.read_csv(CSV_PATH, dtype=str)
    if 'open_time_ms' in df_existing.columns:
        df_existing['open_time_ms'] = pd.to_numeric(df_existing['open_time_ms'], errors='coerce')
        df_existing = df_existing.dropna(subset=['open_time_ms'])
        last_ts = pd.to_datetime(df_existing['open_time_ms'].max(), unit='ms')
        start_date = (last_ts + timedelta(days=1)).date()
        print(f'Found existing data up to {last_ts}. Fetching from {start_date}...')
    else:
        df_existing = None
        print('Existing CSV has wrong format. Re-downloading all data...')
else:
    df_existing = None
    print(f'No existing data. Downloading from {start_date} to {end_date}...')

# Download day by day
all_dfs = []
current = start_date
yesterday = (datetime.now(timezone.utc) - timedelta(days=1)).date()
actual_end = min(end_date, yesterday)

total_days = (actual_end - current).days + 1
if total_days > 0:
    print(f'Downloading {total_days} days of data...')
    
    failed_days = []
    for i in range(total_days):
        date_str = current.strftime('%Y-%m-%d')
        df_day = download_binance_vision_day(SYMBOL, INTERVAL, date_str)
        if df_day is not None and len(df_day) > 0:
            all_dfs.append(df_day)
        else:
            failed_days.append(date_str)
        
        if (i + 1) % 30 == 0 or (i + 1) == total_days:
            print(f'  Progress: {i+1}/{total_days} days ({date_str})')
        
        current += timedelta(days=1)
    
    if failed_days:
        print(f'  Warning: {len(failed_days)} days failed/missing')
else:
    print('Data is already up to date!')

# Merge with existing and save
if all_dfs:
    df_new = pd.concat(all_dfs, ignore_index=True)
    if df_existing is not None:
        # Ensure df_existing is also string dtype for safe concat
        df_existing = df_existing.astype(str)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new
    
    # CRITICAL: Force numeric conversion BEFORE sort
    for col in KLINE_COLUMNS:
        if col in df_combined.columns:
            df_combined[col] = pd.to_numeric(df_combined[col], errors='coerce')
    
    # Drop rows with bad timestamps
    df_combined = df_combined.dropna(subset=['open_time_ms'])
    df_combined = df_combined.drop_duplicates(subset=['open_time_ms'], keep='last')
    df_combined = df_combined.sort_values('open_time_ms').reset_index(drop=True)
    
    # Drop useless columns
    df_combined = df_combined.drop(columns=['close_time_ms', 'ignore'], errors='ignore')
    
    # Save
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
    df_combined.to_csv(CSV_PATH, index=False)
    print(f'\nSaved {len(df_combined):,} bars to {CSV_PATH}')
    print(f'Columns: {list(df_combined.columns)}')
    
    # Verify taker data exists
    taker_sum = df_combined['taker_buy_base_asset_volume'].sum()
    trades_sum = df_combined['number_of_trades'].sum()
    print(f'\n  TAKER BUY VOLUME TOTAL: {taker_sum:,.0f} BTC')
    print(f'  TOTAL TRADES: {trades_sum:,.0f}')
    print('\n  Real microstructure data is now available!')
else:
    print('No new data to download.')

In [ ]:
from scalp2.data.preprocessing import load_binance_csv, resample_ohlcv

# Load the FULL 12-column CSV from Google Drive
CSV_PATH = '/content/drive/MyDrive/scalp2/data/btcusdt_15min_full.csv'
if not os.path.exists(CSV_PATH):
    CSV_PATH = '/content/drive/MyDrive/scalp2/data/btcusdt_15min.csv'
    print('Warning: Using old CSV (may lack taker data). Run download cell above first!')

df_15m = load_binance_csv(CSV_PATH)

print(f'15m: {len(df_15m):,} bars')
print(f'Columns: {list(df_15m.columns)}')
print(f'Range: {df_15m.index[0]} to {df_15m.index[-1]}')

# Verify microstructure columns
has_taker = 'taker_buy_base_vol' in df_15m.columns
has_trades = 'num_trades' in df_15m.columns
print(f'\ntaker_buy_base_vol: {"FOUND" if has_taker else "MISSING"}')
print(f'num_trades: {"FOUND" if has_trades else "MISSING"}')

if has_taker:
    ratio = (df_15m['taker_buy_base_vol'] / (df_15m['volume'] + 1e-10)).mean()
    print(f'\nTaker Buy Ratio (mean): {ratio:.4f}')

df_15m.head()

In [ ]:
# Resample to 1H and 4H from 15m
df_1h = resample_ohlcv(df_15m, '1h')
df_4h = resample_ohlcv(df_15m, '4h')

print(f'1h: {len(df_1h):,} bars')
print(f'4h: {len(df_4h):,} bars')

# Verify taker data survived resampling
for name, d in [('1h', df_1h), ('4h', df_4h)]:
    if 'taker_buy_base_vol' in d.columns:
        print(f'{name} taker_buy_base_vol: preserved')
    else:
        print(f'{name} taker_buy_base_vol: MISSING')

In [ ]:
# Clean all timeframes (gap-fill, validate, optimize dtypes)
from scalp2.data.preprocessing import clean_ohlcv

data = {'15m': df_15m, '1h': df_1h, '4h': df_4h}

for tf in data:
    data[tf] = clean_ohlcv(data[tf], tf)
    print(f'{tf} after cleaning: {len(data[tf]):,} bars')

In [ ]:
# Save cleaned data to Google Drive
for tf, df in data.items():
    path = f'{config.data.processed_dir}/BTC_USDT_{tf}_clean.parquet'
    df.to_parquet(path)
    print(f'Saved {path}  ({len(df):,} bars)')

print('\nData preparation complete.')